In [2]:
import torch
from dataset import create_dataset
from model.UNet import UNet
from model.mlp import ScoreNetwork
from utils.engine import GaussianDiffusionTrainer
from utils.tools import train_one_epoch, load_yaml
from utils.callbacks import ModelCheckpoint

In [3]:
config = load_yaml("config.yml", encoding="utf-8")

In [4]:
cp = torch.load(config["consume_path"])

In [ ]:
# Reverse sampling, ddpm style
def reverse_sample_learned_ddpm(score_model, x_t, num_samples, num_steps, device='cpu'):
    d = 2  # Data dimension
    #x_T = torch.randn((num_samples, d), device=device)
    #x_t = x_T.clone().to(device)

    steps = num_steps + 1
    t = torch.linspace(1.0, 0.0, steps)  # values from 0 to 1

    # Compute alpha_bar_t using cosine schedule
    alpha_bars = f_alpha_bar(t, device=device) # Normalize to start at 1

    # Compute betas
    betas = 1 - (alpha_bars[:-1] / alpha_bars[1:])
    betas = torch.clip(betas, min=1e-5, max=0.999)

    alpha_bars = alpha_bars[1:]

    for t in range(num_steps):

        t_tensor = torch.tensor([ (num_steps - t) / num_steps]).repeat(num_samples, 1).to(device)  # Shape (num_samples, 1)
        with torch.no_grad():
            score_x = score_model(x_t, t_tensor).to(device)  # Use the learned score
        noise = torch.randn_like(x_t).to(device)
        x_t = 1 / (1 - betas[t]) ** 0.5 * (x_t + betas[t] * score_x) +  betas[t] ** 0.5 * noise
        # Breaks graph to save memory
        x_t.detach()
        # x_t.requires_grad = True
    
    return x_t.to(device)



In [15]:
device='cpu'
n_samples=1000
d = 2
n_steps = 300
x_T = torch.randn((n_samples, 2), device=device)
x_t = x_T.clone().to(device)

model = ScoreNetwork()
model.load_state_dict(cp["model"])

ds_c = reverse_sample_learned_ddpm(model, x_t, n_samples, 
                                       n_steps, device=device).detach()

NameError: name 'f_alpha_bar' is not defined